# H2 — SOFR Rate Trajectory per Roll Window

In [1]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)

# SEAGATE required for E notebooks


In [2]:
sofr_candidates = list(SEAGATE_DIR.glob('SOFR*.csv')) + list(SEAGATE_DIR.glob('sofr*.csv'))
if not sofr_candidates:
    # fallback: try to fetch from FRED (public, no key)
    import urllib.request, io
    url = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=SOFR'
    with urllib.request.urlopen(url) as r:
        sofr = pd.read_csv(io.BytesIO(r.read()), parse_dates=['DATE'], index_col='DATE')
    sofr.columns = ['rate']
    print('Fetched SOFR from FRED')
else:
    sofr = pd.read_csv(sofr_candidates[0], parse_dates=[0], index_col=0)
    sofr.columns = ['rate']
    print(f'Loaded {sofr_candidates[0].name}')

fig, ax = plt.subplots(figsize=(12, 5))
for wk, wm in WINDOWS_META.items():
    roll_start = pd.Timestamp(wm['roll_start'])
    window = sofr.loc[roll_start - pd.Timedelta(days=10) : roll_start + pd.Timedelta(days=1), 'rate']
    window = window.dropna()
    if window.empty: continue
    x = range(len(window))
    ax.plot(x, window.values, marker='o', label=wk, color=WIN_COLORS[wk], lw=1.5)
    delta = window.iloc[-1] - window.iloc[0]
    ax.annotate(f'Δ={delta:+.3f}%', xy=(len(window)-1, window.iloc[-1]),
                xytext=(len(window)-1+0.2, window.iloc[-1]), fontsize=8)
ax.set_xlabel('Days before roll week start')
ax.set_ylabel('SOFR Rate (%)')
ax.set_title('SOFR Rate 10 Days Pre-Roll per Window')
ax.legend()
save_fig(fig, 'H2_sofr_trajectory.png')


Loaded SOFR.csv
  Saved --> figures/H2_sofr_trajectory.png
